# 03 - Localization Pipeline and Method Comparison
This notebook evaluates the localization capabilities of the trained model. It compares a custom LSE-aware Manual CAM, standard Grad-CAM, and Grad-CAM++ to visually assess how well the model localizes illicit waste without pixel-level annotations. It requires the pre-trained weights from the previous training step.

In [ ]:
# State Restoration: Load Model Weights[cite: 1]
model.load_state_dict(torch.load(CFG["model_path"], map_location=CFG["device"]))
model.eval()
print("Loaded best model weights from disk.")

In [ ]:
# Cell 6: Localization Pipeline[cite: 1]
target_layers = [model.layer4[-1]]
cam_extractor = GradCAMPlusPlus(model=model, target_layers=target_layers)

def get_refined_heatmap(input_tensor):
    targets = [ClassifierOutputTarget(BEFORE_IDX)]
    raw_cam = cam_extractor(input_tensor=input_tensor, targets=targets)[0, :]
    refined = cv2.GaussianBlur(raw_cam, CFG["smooth_kernel"], 0)
    denom = (refined.max() - refined.min()) + 1e-8
    refined = (refined - refined.min()) / denom
    return refined

def adaptive_dilation(mask, heatmap_area_ratio):
    if heatmap_area_ratio < 0.02:
        kernel_size = 3
    elif heatmap_area_ratio < 0.08:
        kernel_size = 5
    else:
        kernel_size = 3
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    return cv2.dilate(mask, kernel, iterations=1)

def get_final_mask(heatmap, model_confidence=1.0):
    h, w = heatmap.shape
    margin_h, margin_w = int(h * 0.05), int(w * 0.05)
    spatial_mask = np.zeros_like(heatmap)
    spatial_mask[margin_h:h-margin_h, margin_w:w-margin_w] = 1.0
    heatmap = heatmap * spatial_mask
    
    heatmap_prob = heatmap / (heatmap.sum() + 1e-8)
    heatmap_entropy = -np.sum(heatmap_prob * np.log(heatmap_prob + 1e-8))
    normalized_entropy = heatmap_entropy / np.log(heatmap.size)
    
    if normalized_entropy > 0.97 and model_confidence > 0.98:
        return np.zeros_like(heatmap, dtype=np.uint8)
    
    heatmap_8bit = np.uint8(255 * heatmap)
    val, mask = cv2.threshold(heatmap_8bit, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if val < (255 * 0.25):
        _, mask = cv2.threshold(heatmap_8bit, int(255 * 0.25), 255, cv2.THRESH_BINARY)
    
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    pre_dilate_ratio = (mask > 0).mean()
    mask = adaptive_dilation(mask, pre_dilate_ratio)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_filled = np.zeros_like(mask)
    cv2.drawContours(mask_filled, contours, -1, 255, thickness=cv2.FILLED)
    mask = mask_filled

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n_labels <= 1: return mask
    
    areas = stats[1:, cv2.CC_STAT_AREA]
    max_area = np.max(areas)
    significant_indices = np.where(areas > (max_area * 0.08))[0] + 1
    
    final_mask = np.zeros_like(mask)
    for idx in significant_indices:
        final_mask[labels == idx] = 255
            
    return final_mask

print(f"Localization engine ready. Device: {CFG['device']}")

In [ ]:
# Cell 6.5: Original CAM from Scratch (Weight-Based)[cite: 1]
class ManualCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.hook_handle = self.target_layer.register_forward_hook(self.save_activations)

    def save_activations(self, module, input, output):
        self.activations = output.detach()

    def generate(self, input_tensor, class_idx=1, r=5):
        self.model.eval()
        with torch.no_grad():
            _ = self.model(input_tensor)
    
        fc_weights   = self.model.fc.weight.data
        class_weights_c = fc_weights[class_idx]
        A = self.activations[0]
        c, h, w = A.shape
    
        A_flat = A.view(c, -1)
        A_max  = A_flat.max(dim=1, keepdim=True).values
        softmax_w = torch.softmax(r * (A_flat - A_max), dim=1)
        softmax_w = softmax_w.view(c, h, w)
    
        cam = torch.einsum('c, chw, chw -> hw', class_weights_c, A, softmax_w)
        cam = torch.maximum(cam, torch.tensor(0.0, device=cam.device))
        cam = cam.cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def remove_hook(self):
        self.hook_handle.remove()

manual_cam_engine = ManualCAM(model, model.layer4[-1])
print("LSE-Aware Spatial CAM (Custom) initialized.")

In [ ]:
# Cell 7: Triple-Method Benchmark[cite: 1]
manual_engine = manual_cam_engine 
target_layers = [model.layer4[-1]]
grad_cam_engine = GradCAM(model=model, target_layers=target_layers)
grad_cam_plus_engine = GradCAMPlusPlus(model=model, target_layers=target_layers)

def get_all_heatmaps(input_tensor, class_idx):
    h_manual = manual_engine.generate(input_tensor, class_idx=class_idx)
    h_manual = cv2.resize(h_manual, (CFG["img_size"], CFG["img_size"]))
    h_manual = cv2.GaussianBlur(h_manual, CFG["smooth_kernel"], 0)
    
    targets = [ClassifierOutputTarget(class_idx)]
    h_grad = grad_cam_engine(input_tensor=input_tensor, targets=targets)[0, :]
    h_grad = cv2.GaussianBlur(h_grad, CFG["smooth_kernel"], 0)
    
    h_plus = grad_cam_plus_engine(input_tensor=input_tensor, targets=targets)[0, :]
    h_plus = cv2.GaussianBlur(h_plus, CFG["smooth_kernel"], 0)
    
    h_manual = (h_manual - h_manual.min()) / (h_manual.max() - h_manual.min() + 1e-8)
    h_grad = (h_grad - h_grad.min()) / (h_grad.max() - h_grad.min() + 1e-8)
    h_plus = (h_plus - h_plus.min()) / (h_plus.max() - h_plus.min() + 1e-8)
    
    return h_manual, h_grad, h_plus

def calculate_iou(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return intersection / union

samples = [i for i, (p, l) in enumerate(test_set.samples) if l == BEFORE_IDX]
selected = random.sample(samples, 3)

fig, axes = plt.subplots(len(selected), 6, figsize=(26, 4 * len(selected)))
plt.subplots_adjust(wspace=0.1, hspace=0.4)
col_titles = ["Input", "Manual CAM\n(Weights)", "Grad-CAM\n(Gradients)", "Grad-CAM++\n(Pixel-Weights)", "Expert GT", "Final Mask"]

for row, s_idx in enumerate(selected):
    path, _ = test_set.samples[s_idx]
    img_pil = Image.open(path).convert('RGB')
    input_tensor = eval_transforms(img_pil).unsqueeze(0).to(CFG["device"])
    
    img_np = np.array(img_pil.resize((CFG["img_size"], CFG["img_size"]))).astype(np.float32) / 255.0

    h_man, h_grad, h_plus = get_all_heatmaps(input_tensor, BEFORE_IDX)
    
    with torch.no_grad():
        out = model(input_tensor)
        conf = F.softmax(out, dim=1)[0][BEFORE_IDX].item()
    pred_mask = get_final_mask(h_plus, model_confidence=conf)
    
    gt_path = Path(str(path).replace('images', 'masks').replace('.jpg', '.png'))
    if not gt_path.exists(): 
        gt_path = Path(str(path).replace('images', 'masks').replace('.jpg', '.jpg'))
    
    gt_binary = np.zeros((CFG["img_size"], CFG["img_size"]), dtype=bool)
    if gt_path.exists():
        gt_img = Image.open(gt_path).convert('L').resize((CFG["img_size"], CFG["img_size"]), Image.NEAREST)
        gt_binary = np.array(gt_img) > 128

    iou = calculate_iou(pred_mask > 0, gt_binary)

    axes[row, 0].imshow(img_np)
    axes[row, 1].imshow(show_cam_on_image(img_np, h_man, use_rgb=True))
    axes[row, 2].imshow(show_cam_on_image(img_np, h_grad, use_rgb=True))
    axes[row, 3].imshow(show_cam_on_image(img_np, h_plus, use_rgb=True))
    axes[row, 4].imshow(gt_binary, cmap='gray')
    axes[row, 5].imshow(pred_mask, cmap='gray')
    
    axes[row, 5].set_title(f"IoU: {iou:.4f}", fontweight='bold', color='green' if iou > 0.2 else 'red')
    
    if row == 0:
        for c, t in enumerate(col_titles): 
            axes[row, c].set_title(t, fontweight='bold', fontsize=12)
    for ax in axes[row]: 
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("Weakly Supervised Localization: Three-Way Methodological Comparison", fontsize=20, y=1.02)
plt.show()